# Day 7 Lab 04: Bronze Raw Ingestion

        **Layer:** Bronze  
        **Python reference:** `Lab_Files/labs/lab_04_bronze_raw_ingestion.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read all raw order-event batches.
- Add Bronze metadata columns.
- Write the raw Bronze Parquet table and inspect counts by source file.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import LAKE_DIR, OUTPUT_DIR, clean_lab_dir, ensure_output_dirs, read_order_events, require_source_data, spark_session, with_bronze_metadata, write_csv_dir

## 2. Start Spark

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook04BronzeRawIngestion")

## 3. Read Raw Order Events

In [ ]:
raw_orders = read_order_events(spark)
raw_orders.printSchema()
raw_orders.show(10, truncate=False)

## 4. Add Bronze Metadata

In [ ]:
bronze = with_bronze_metadata(raw_orders, "notebook-full-load-001")
bronze.select("event_id", "order_id", "_source_file", "_ingestion_batch_id", "_bronze_ingested_at").show(10, truncate=False)

## 5. Write Bronze

In [ ]:
bronze_path = clean_lab_dir(LAKE_DIR / "bronze" / "orders_raw")
bronze.write.mode("overwrite").parquet(str(bronze_path))
print(f"Bronze path: {bronze_path}")

## 6. Audit Counts by Source File

In [ ]:
by_file = spark.read.parquet(str(bronze_path)).groupBy("_source_file").count().orderBy("_source_file")
by_file.show(truncate=False)
write_csv_dir(by_file, OUTPUT_DIR / "notebook_04_bronze_counts_by_file")
print(f"Bronze rows written: {bronze.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()